In [1]:
from dotenv import load_dotenv
from supabase import create_client
import os
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[1]
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError("SUPABASE_URL and SUPABASE_KEY must be defined in the .env file.")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Connected to Supabase")

Connected to Supabase


## Load new Templates

In [4]:
from glob import glob

In [ ]:
data_path  = PROJECT_ROOT / 'dev/data/new_templates' 

files =  glob(str(data_path)+'/*.yaml')

['C:\\Users\\Home\\Desktop\\projetos\\mark-me-down\\dev\\data\\new_templates\\class_notes.yaml',
 'C:\\Users\\Home\\Desktop\\projetos\\mark-me-down\\dev\\data\\new_templates\\daily_journal.yaml',
 'C:\\Users\\Home\\Desktop\\projetos\\mark-me-down\\dev\\data\\new_templates\\recipe.yaml',
 'C:\\Users\\Home\\Desktop\\projetos\\mark-me-down\\dev\\data\\new_templates\\resume.yaml',
 'C:\\Users\\Home\\Desktop\\projetos\\mark-me-down\\dev\\data\\new_templates\\todo_list.yaml',
 'C:\\Users\\Home\\Desktop\\projetos\\mark-me-down\\dev\\data\\new_templates\\weekly_journal.yaml']

In [ ]:
# Open and parse yaml files
import yaml

templates = []
for file in files:
    with open(file, 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)
        templates.append(data)

## Get Embedding Text field

In [ ]:

# Building Template Profile semantic doc 
from textwrap import dedent

def note_profile_to_embedding_text(profile: dict) -> str:
    """Serialize a NoteProfile into a deterministic text for embeddings."""

    def yaml_list(items: list[str]) -> str:
        return "\n".join(f"  - {item}" for item in items)

    parts = []
    
    # Map of output topic header to the profile attribute name
    topics = [
        ('name','name'),
        ('description','description'),
        ('instructions','instructions'),
        ('category','category'),
        ('tags','tags'),
        ('version','version'),
        # ('preview_markdown','preview_markdown'), #Example
        # ('template_markdown','template_markdown'), # Template Model
        ('purpose','purpose'),
        ('sections','sections'),
        ('organization_structure','organization_structure'),
        ('style','style'),
        ( 'raw_note_text','raw_note_text')
    ]
    
    for topic_name, attr_name in topics:
        if topic_name in profile:
            val = profile[attr_name]
            # Ensure the value exists, is not an empty string, and (if list) is not empty
            if val is not None and val != "" and (not isinstance(val, list) or len(val) > 0):
                formatted_list = yaml_list(val) if isinstance(val, list) else f"  - {val}"
                parts.append(f"{topic_name}:\n{formatted_list}")
                
    return "\n\n".join(parts).strip()


In [ ]:
for note in templates:

    note['embedding_text'] = note_profile_to_embedding_text(note) 

# Print or inspect the first result
print(note['embedding_text'])


### Vectorize embedding text

In [13]:
docs = [note['embedding_text'] for note in templates]
docs

['name:\n  - Class Notes\n\ndescription:\n  - A structured template for capturing notes during lectures, workshops, university classes, online courses, training sessions, bootcamps and educational presentations. It emphasizes separating factual class content from personal reflections, questions and future study actions. The template supports active learning, later revision and long-term knowledge retention.\n\n\ninstructions:\n  - Record information as the class progresses. Capture concepts rather than verbatim transcription whenever possible. Use the "Questions & Doubts" section for unresolved topics, "Insights" for important realizations and "Quick Thoughts" for spontaneous observations. Summarize the lecture before finishing the note.\n\n\ncategory:\n  - Education\n\ntags:\n  - class\n  - class_notes\n  - lecture\n  - lecture_notes\n  - course\n  - course_notes\n  - study\n  - study_notes\n  - education\n  - learning\n  - university\n  - college\n  - school\n  - workshop\n  - traini

In [14]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.embed_content(
    model="gemini-embedding-001",
    contents=docs,
    config=types.EmbedContentConfig(
        output_dimensionality=1536
    ),
)


In [15]:
len([i.values for  i in response.embeddings])

6

In [16]:
for n,note in enumerate(templates):

    note['embedding_vector'] = response.embeddings[n].values




## Save Data locally  

In [17]:

# Path to the data file
import jsonlines

jsonl_path = PROJECT_ROOT / "dev" / "data" / "new-set-templates-enriched2.jsonl"


# Open the file and write all dictionaries at once
with jsonlines.open(jsonl_path, mode="w") as writer:
    writer.write_all(templates)


## Upsert New Templates to Supabase templates table 

In [18]:
for template in templates:
    print(f"Upserting {template['id']}...")
    result = (
        supabase
        .table("templates")
        .upsert(template, on_conflict="id")
        .execute()
    )
    print(f"✓ {template['id']} upserted.")

Upserting class_notes...
✓ class_notes upserted.
Upserting daily_journal...
✓ daily_journal upserted.
Upserting recipe...
✓ recipe upserted.
Upserting resume...
✓ resume upserted.
Upserting todo_list...
✓ todo_list upserted.
Upserting weekly_journal...
✓ weekly_journal upserted.
